# Signal Soup: Sharedness vs. Code Alignment

**A computational model exploring emergent communication through semiotic sign functions**

Andrea Valle  
University of Turin

---

This notebook provides an interactive tutorial and exploration of the Signal Soup model, demonstrating how communicative alignment emerges among agents without complete code sharing.

## Setup and Imports

First, let's import the necessary libraries:

In [ ]:
# Core imports
from signalSoup import CommunicativeAgent, Band
import graphviz

# Visualization and analysis
import matplotlib.pyplot as plt
import numpy as np
from tqdm import tqdm

# IPython utilities for interactive display
from IPython import display
from IPython.display import clear_output
import time

# Set plotting style
plt.style.use('default')
%matplotlib inline

## 1. Theoretical Background

### The Communicative Environment (CEnv)

A **communicative environment (CEnv)** is made up of signals. CEnv is dynamic - it can be modified by agents and potentially by non-agentive events.

**Signals** are represented as three-letter words from 26 lowercase letters, yielding 26³ = 17,576 possible signals.

### Communicative Agents (CAgs)

A **communicative agent (CAg)** interacts with the environment by:
- **Retrieving** signals from CEnv
- **Processing** them through detector-effector mappings
- **Dispatching** new signals back to CEnv

### Sign Functions

Each agent has:
- **Detectors**: Signal patterns it can recognize
- **Effectors**: Signal patterns it can produce  
- **Sign Functions**: Learned mappings d → e (detector → effector)
- **Relevance Values**: Integer scores (0-10) indicating strength
- **Limited Memory**: Maximum 10 sign functions

### Agent Behavior

When an agent inspects CEnv:

1. **Match found**: Detector matches signal → trigger effector, replace signal, increment relevance
2. **No match**: Create new sign function from random signals, assign relevance = 1

## 2. Single Agent Exploration

### 2.1 Creating an Agent with Custom Sign Functions

In [ ]:
# Create agent and environment
a = CommunicativeAgent()
sr = 'xvg hju xvg bzi'.split()

# Set up sign functions manually
a.detectors = 'xvg bzi xvg'.split()
a.effectors = 'eee fff ggg'.split()
a.values = [0, 0, 0]

print("Environment:", sr)
print("Detectors:", a.detectors)
print("Effectors:", a.effectors)
print("Initial values:", a.values)

In [ ]:
# Agent perceives and responds
a.retrieve(sr)
matches = a.count_matches()

print("Matches found:", matches)
print("\nBefore response:", a.comm_env)
print("Values before:", a.values)

a.respond(matches)

print("\nAfter response:", a.comm_env)
print("Values after:", a.values)

### 2.2 Visualizing Agent Topology

In [ ]:
# Create agent with random sign functions
agent = CommunicativeAgent(sign_function_num=5, word_length=3, memory_length=10)
agent.make_sign_functions()

# Simulate interactions
test_env = [agent.make_word(3) for _ in range(10)]
for _ in range(20):
    agent.retrieve(test_env)
    matches = agent.count_matches()
    agent.check_and_act(matches)
    test_env = agent.comm_env.copy()

# Visualize
agent.plot_agent()

In [ ]:
# Topology visualization (red = signals that are both detectors and effectors)
agent.plot_topology()

## 3. Band Dynamics

### 3.1 Creating and Running a Band

In [ ]:
# Create band
band = Band(num_agents=20, word_length=3, memory_length=10, seed=42)

print(f"Band: {len(band.agents)} agents")
print(f"Environment size: {len(band.comm_env)} signals")
print(f"Unique types: {len(set(band.comm_env))}")

### 3.2 Simulation Over Time

In [ ]:
# Run simulation
print("Running 2000 turns...\n")

for i in tqdm(range(2000)):
    band.turn()
    if (i + 1) % 500 == 0:
        types = len(set(band.comm_env))
        tokens = len(band.comm_env)
        print(f"Turn {i+1}: {types} types / {tokens} tokens = {types/tokens:.3f}")

print(f"\nFinal: {len(set(band.comm_env))}/{len(band.comm_env)} type/token")

### 3.3 Visualizing the Band

In [ ]:
band.plot_band()

## 4. Sharedness Analysis

In [ ]:
# Measure sharedness
sf_dict = band.get_sign_function_dict()
shared = {sf: count for sf, count in sf_dict.items() if count > 1}

print(f"Total unique sign functions: {len(sf_dict)}")
print(f"Shared (2+ agents): {len(shared)} ({100*len(shared)/len(sf_dict):.1f}%)")

print("\nMost shared:")
for (d, e), c in sorted(shared.items(), key=lambda x: x[1], reverse=True)[:5]:
    print(f"  {d} → {e}: {c} agents")

In [ ]:
# Visualize sharedness (red = shared)
band.plot_sharedness(sf_dict)

## 5. Understanding Convergence: Why ~0.33?

Before analyzing the time-series data, let's understand a key theoretical property of Signal Soup:

### The Type-Token Convergence Phenomenon

One of the most striking emergent properties is that **all band sizes converge to a type-token ratio of approximately 0.33** (roughly 3 tokens per signal type), regardless of the number of agents.

#### Why Does This Happen?

The convergence emerges from the interaction of several mechanisms:

**1. Reinforcement Learning**  
Sign functions that successfully match signals get reinforced (relevance +1). This creates a **positive feedback loop**: frequently used signals become more likely to be produced, which makes them more likely to match, which makes them more likely to be produced again.

**2. Limited Memory**  
Each agent can store only ~10 sign functions. This constraint forces agents to:
- Focus on signals that appear frequently in CEnv
- Replace rarely-used sign functions with new ones
- Converge toward a common subset of signals

**3. Stigmergic Coordination**  
Agents coordinate *indirectly* through the shared environment:
- Agent A produces signal 'xyz'
- Agent B notices 'xyz' appears frequently → creates sign function for 'xyz'
- Agent B also produces 'xyz' → reinforces the pattern
- Result: convergent behavior without direct communication

**4. Environment Size = Band Size**  
The communicative environment has the same number of slots as there are agents. This creates a natural balance:
- Too much diversity → hard to match, sign functions don't get reinforced
- Too little diversity → easy to match, but limited expressive capacity

#### Why Specifically ~0.33 (3 tokens per type)?

The value 0.33 represents an **equilibrium** between:

**Diversity vs. Redundancy**
- **Too high ratio** (e.g., 0.8): Many different signals, few matches, weak reinforcement
- **Too low ratio** (e.g., 0.1): Only a few signal types, high redundancy, limited communication capacity
- **~0.33**: Optimal balance where agents can reliably match signals while maintaining sufficient variety

**Mathematical Intuition**
With:
- Memory capacity ≈ 6-7 sign functions (observed average)
- Environment size = n
- Multiple agents sharing overlapping sign functions

The equilibrium emerges where each signal type appears approximately 3 times in the environment, yielding a type-token ratio of 1/3.

#### Universality

The remarkable property is that this ratio is **universal**:
- Band of 10 agents → converges to 0.33
- Band of 100 agents → converges to 0.33
- Different word lengths → similar convergence

Only the **time to convergence** changes (larger bands take longer), but the **final equilibrium** is the same.

#### Statistical Nature

The convergence is a **statistical property**:
- Individual runs show stochastic fluctuations
- Averaging over multiple runs reveals the underlying attractor
- Variance decreases over time as the system stabilizes

This is why we average over multiple runs: not because convergence is uncertain, but because it's a **robust statistical property** of the system, analogous to thermodynamic equilibrium in physics.

---

Now let's observe this convergence in action through time-series analysis.

## 5a. Time-Series Analysis: Observing Convergence

### 5.1 Type-Token Ratio Convergence

We'll track how the type-token ratio converges over time, averaging over multiple runs for robustness.

In [ ]:
# Type-token convergence with multiple runsnum_agents = 30num_runs = 20  # Average over 20 runsnum_turns = 5000sample_interval = 50print(f"Type-Token Convergence Analysis")print(f"Band size: {num_agents} agents")print(f"Runs: {num_runs}")print(f"Turns per run: {num_turns}\n")all_trajectories = []turns_sampled = []  # Fixed: moved outside the loopfor run in tqdm(range(num_runs), desc="Runs"):    band_ts = Band(num_agents=num_agents, seed=run)        trajectory = []        for turn in range(num_turns):        band_ts.turn()        if turn % sample_interval == 0:            ratio = len(set(band_ts.comm_env)) / len(band_ts.comm_env)            trajectory.append(ratio)            if run == 0:  # Only save turns once                turns_sampled.append(turn)        all_trajectories.append(trajectory)# Calculate mean and stdall_trajectories = np.array(all_trajectories)mean_trajectory = np.mean(all_trajectories, axis=0)std_trajectory = np.std(all_trajectories, axis=0)# Plotplt.figure(figsize=(10, 5))# Plot mean with confidence bandplt.plot(turns_sampled, mean_trajectory, linewidth=2, label='Mean', color='steelblue')plt.fill_between(turns_sampled,                  mean_trajectory - std_trajectory,                 mean_trajectory + std_trajectory,                 alpha=0.3, color='steelblue', label='±1 std')# Target lineplt.axhline(1/3, color='red', linestyle='--', linewidth=2, label='Target (0.333)')plt.xlabel('Turns', fontsize=12)plt.ylabel('Type-Token Ratio', fontsize=12)plt.title(f'Type-Token Ratio Convergence (averaged over {num_runs} runs)',           fontsize=14, fontweight='bold')plt.legend()plt.grid(alpha=0.3)plt.tight_layout()plt.show()print(f"\nInitial type-token ratio: {mean_trajectory[0]:.3f} ± {std_trajectory[0]:.3f}")print(f"Final type-token ratio: {mean_trajectory[-1]:.3f} ± {std_trajectory[-1]:.3f}")print(f"Target: 0.333 (3 tokens per type)")print(f"\n✓ Convergence clearly visible with reduced variance")

#### Observations on Convergence

Notice how the plot demonstrates the theoretical principles:

1. **Initial Phase** (0-500 turns): Rapid decrease from 1.0 as agents start creating sign functions and signals begin to repeat

2. **Transition Phase** (500-2000 turns): The system explores different configurations, variance is high

3. **Equilibrium Phase** (2000+ turns): Convergence to ~0.33, variance decreases significantly

4. **Statistical Robustness**: The confidence band (shaded area) shows that while individual runs fluctuate, the mean trajectory reveals the underlying attractor at 0.33

This is a **statistical equilibrium**, not a deterministic fixed point—similar to how gas molecules reach thermodynamic equilibrium despite constant microscopic fluctuations.

### 5.2 Sign Functions Per Agent

In [ ]:
# Sign function growth with multiple runsnum_agents_sf = 40num_runs_sf = 20num_turns_sf = 10000sample_interval_sf = 100print(f"Sign Function Growth Analysis")print(f"Band size: {num_agents_sf} agents")print(f"Runs: {num_runs_sf}")print(f"Turns per run: {num_turns_sf}\n")all_sf_trajectories = []turns_sampled_sf = []  # Fixed: moved outside the loopfor run in tqdm(range(num_runs_sf), desc="Runs"):    band_sf = Band(num_agents=num_agents_sf, seed=run)        sf_trajectory = []        for turn in range(num_turns_sf):        band_sf.turn()        if turn % sample_interval_sf == 0:            avg_sf = np.mean([len(agent.detectors) for agent in band_sf.agents])            sf_trajectory.append(avg_sf)            if run == 0:                turns_sampled_sf.append(turn)        all_sf_trajectories.append(sf_trajectory)# Calculate mean and stdall_sf_trajectories = np.array(all_sf_trajectories)mean_sf = np.mean(all_sf_trajectories, axis=0)std_sf = np.std(all_sf_trajectories, axis=0)# Plotplt.figure(figsize=(10, 5))plt.plot(turns_sampled_sf, mean_sf, linewidth=2, color='darkgreen', label='Mean')plt.fill_between(turns_sampled_sf,                 mean_sf - std_sf,                 mean_sf + std_sf,                 alpha=0.3, color='darkgreen', label='±1 std')plt.axhline(10, color='red', linestyle='--', linewidth=2, label='Memory limit')plt.xlabel('Turns', fontsize=12)plt.ylabel('Avg Sign Functions/Agent', fontsize=12)plt.title(f'Sign Function Growth (averaged over {num_runs_sf} runs)',           fontsize=14, fontweight='bold')plt.legend()plt.grid(alpha=0.3)plt.ylim(0, 11)plt.tight_layout()plt.show()print(f"\nInitial avg SF/agent: {mean_sf[0]:.2f} ± {std_sf[0]:.2f}")print(f"Final avg SF/agent: {mean_sf[-1]:.2f} ± {std_sf[-1]:.2f}")print(f"Memory limit: 10")print(f"\n→ Agents stabilize around {mean_sf[-1]:.1f} sign functions")print(f"   (well below the memory limit of 10)")

## 6. Population Size Effects

How does band size affect convergence dynamics? We'll test this with two approaches:

1. **Quick test** (2-3 minutes) - For rapid exploration
2. **Full analysis** (10-15 minutes) - Reproduces complete results with 100 runs per band size

### 6.1 Quick Test (2-3 minutes)

A simplified version to quickly see the convergence pattern.

In [ ]:
# Quick test with fewer parameters
band_sizes_quick = [20, 50, 80]
num_runs_quick = 10
num_turns_quick = 5000
sample_interval = 100

results_quick = []

print("Quick population analysis...")
print(f"Band sizes: {band_sizes_quick}")
print(f"Runs per size: {num_runs_quick}")
print(f"Estimated time: 2-3 minutes\n")

for size in band_sizes_quick:
    print(f"Testing band size {size}...")
    
    runs = []
    for run in range(num_runs_quick):
        band_temp = Band(num_agents=size, seed=run)
        
        trajectory = []
        for turn in range(num_turns_quick):
            band_temp.turn()
            if turn % sample_interval == 0:
                ratio = len(set(band_temp.comm_env)) / len(band_temp.comm_env)
                trajectory.append(ratio)
        
        runs.append(trajectory)
    
    # Average over runs
    avg_trajectory = np.mean(runs, axis=0)
    results_quick.append(avg_trajectory)

# Plot
plt.figure(figsize=(10, 6))

for i, size in enumerate(band_sizes_quick):
    x = list(range(0, num_turns_quick, sample_interval))
    plt.plot(x, results_quick[i], label=f'n={size}', linewidth=2)

plt.axhline(1/3, color='black', linestyle='--', alpha=0.7, linewidth=2)
plt.xlabel('Turns', fontsize=12)
plt.ylabel('Type-Token Ratio', fontsize=12)
plt.title('Type-Token Convergence - Quick Test', fontsize=14, fontweight='bold')
plt.legend()
plt.grid(alpha=0.3)
plt.ylim(0.2, 1.0)
plt.tight_layout()
plt.show()

print("\n✓ Quick test complete")
print("→ All sizes converge toward ~0.33")
print("\nRun next cell for full analysis with more band sizes and runs")

### 6.2 Full Analysis (10-15 minutes)

⚠️ **Warning**: This cell takes 10-15 minutes to run.

Complete analysis with:
- 10 band sizes (10, 20, 30, ..., 100)
- 100 runs per band size
- Sampling every turn

This reproduces the complete convergence analysis.

In [ ]:
# Full analysis (takes 10-15 minutes)
band_sizes_full = list(range(10, 110, 10))
num_runs_full = 100
num_turns_full = 10000

results_full = []

print("Full population analysis...")
print(f"Band sizes: {band_sizes_full}")
print(f"Runs per size: {num_runs_full}")
print(f"Turns per run: {num_turns_full}")
print(f"Total simulations: {len(band_sizes_full) * num_runs_full}")
print("\n⏱️  Estimated time: 10-15 minutes...\n")

for size in band_sizes_full:
    print(f"Band size {size}...")
    
    runs = []
    for run in tqdm(range(num_runs_full), desc=f"Size {size}", leave=False):
        band_temp = Band(num_agents=size, seed=run)
        
        # Sample every turn for complete trajectory
        trajectory = []
        for turn in range(num_turns_full):
            band_temp.turn()
            ratio = len(set(band_temp.comm_env)) / len(band_temp.comm_env)
            trajectory.append(ratio)
        
        runs.append(trajectory)
    
    # Average over all runs
    avg_trajectory = np.mean(runs, axis=0)
    results_full.append(avg_trajectory)
    print(f"  Final ratio: {avg_trajectory[-1]:.3f}")

# Plot (matching original figure)
plt.figure(figsize=(12, 7))

for i, size in enumerate(band_sizes_full):
    plt.plot(results_full[i], label=f'{size}', linewidth=1.5)

plt.axhline(1/3, color='black', linestyle='--', alpha=0.7, linewidth=2)
plt.xlabel('Turns', fontsize=12)
plt.ylabel('Type-Token Ratio', fontsize=12)
plt.title('Type-Token Convergence for Different Band Sizes (100 runs each)', 
          fontsize=14, fontweight='bold')
plt.legend(loc='upper right', fontsize=10, ncol=2)
plt.grid(alpha=0.3)
plt.ylim(0.2, 1.0)
plt.tight_layout()
plt.show()

print("\n✓ Full analysis complete!")
print("\nKey observations:")
print("- All sizes converge to ~0.33 (3 tokens per type)")
print("- Smaller bands converge faster")
print("- Larger bands show more initial variance but same final convergence")

## 7. Animated Visualization

In [ ]:
# Watch evolution in real-time
band_anim = Band(num_agents=15, seed=999)

print("Animation running (Ctrl+C to stop)...")

try:
    for i in range(100):
        display.display(band_anim.plot_band())
        display.clear_output(wait=True)
        time.sleep(0.3)
        band_anim.turn()
        print(f"Turn {i+1}: {len(set(band_anim.comm_env))} types")
except KeyboardInterrupt:
    print("Stopped")

## 8. Relevance Distribution

In [ ]:
# Collect relevance values
all_values = []
for agent in band.agents:
    all_values.extend(agent.values)

# Plot
plt.figure(figsize=(10, 5))
plt.hist(all_values, bins=range(0, 12), edgecolor='black', alpha=0.7)
plt.xlabel('Relevance Value')
plt.ylabel('Frequency')
plt.title('Distribution of Relevance Values')
plt.xticks(range(0, 11))
plt.grid(alpha=0.3, axis='y')
plt.tight_layout()
plt.show()

print(f"Mean: {np.mean(all_values):.2f}")
print(f"Median: {np.median(all_values):.2f}")

## 9. Summary

### Key Findings:

1. **Universal Type-Token Convergence**: All band sizes converge to ~0.33 (3 tokens per signal type), regardless of population size—a robust emergent property of the system

2. **Sharedness Emergence**: ~26% of sign functions become shared across multiple agents without any centralized coordination

3. **Equilibrium Sign Functions**: Agents stabilize around ~6 sign functions each (well below the memory limit of 10), suggesting a natural equilibrium

4. **Feedback Loop Formation**: Signals that are both detectors and effectors emerge spontaneously, creating self-reinforcing patterns

5. **Loose Alignment Sufficiency**: Communication persists through partial overlap in sign functions rather than complete code convergence

### Theoretical Implications:

**On Convergence:**
- The 0.33 ratio represents a **statistical equilibrium** between diversity and redundancy
- **Stigmergic coordination** through the shared environment drives convergence
- **Reinforcement learning** creates positive feedback loops favoring common signals
- **Limited memory** constrains agents toward a shared subset of signals
- The convergence is **universal** (all sizes) but **statistical** (requires averaging)

**On Communication:**
- **Sharedness ≠ Complete Alignment**: Partial overlap in sign functions is sufficient for stable communication
- **Environment-Mediated Coordination**: Agents influence each other indirectly through CEnv modifications
- **Emergent Meaning**: Signal meaning arises from the network of sign functions, not from pre-established semantics
- **Encyclopedia over Dictionary**: Agents maintain idiosyncratic repertoires (Eco's "encyclopedic" competence) while achieving functional coordination

**On Complex Systems:**
- Signal Soup exhibits properties of **complex adaptive systems**: emergence, self-organization, statistical equilibria
- The model demonstrates how **local interactions** (agent-environment) produce **global patterns** (universal convergence)
- **Stochasticity** at the microscopic level doesn't prevent **deterministic** macroscopic behavior (convergence)

### Scientific Significance:

The type-token convergence to 0.33 is particularly significant because:
- It's **emergent** (not programmed in)
- It's **robust** (consistent across conditions)
- It's **universal** (independent of scale)
- It's **stable** (maintained over time)

This makes it a genuine **law-like regularity** of the system, analogous to phase transitions in physics or population equilibria in ecology.

---

**Repository**: https://github.com/vanderaalle/signal-soup  
**Contact**: andrea.valle@unito.it

**Paper**: Valle, A. (2025). Signal Soup: Sharedness vs. code alignment. *Proceedings of EVOLANG*.


## 10. Exercises

Try these:

1. Change `memory_length` - how does it affect sharedness?
2. Test `word_length=4` or `word_length=2`
3. Vary `comm_env_size`
4. Add random perturbations
5. Create heterogeneous agents

### Example: Memory Size Effects

In [ ]:
memory_sizes = [5, 10, 15, 20]
results_mem = {}

for mem in memory_sizes:
    b = Band(num_agents=20, memory_length=mem, seed=42)
    for _ in range(5000):
        b.turn()
    
    sf = b.get_sign_function_dict()
    shared_pct = 100 * sum(1 for c in sf.values() if c > 1) / len(sf)
    results_mem[mem] = shared_pct
    print(f"Memory {mem}: {shared_pct:.1f}% shared")

plt.figure(figsize=(8, 5))
plt.bar(memory_sizes, [results_mem[m] for m in memory_sizes])
plt.xlabel('Memory Length')
plt.ylabel('Sharedness (%)')
plt.title('Memory Size vs Sharedness')
plt.show()